# FotMob Fixtures Scraper — All Domestic Leagues

Scrapes fixture lists for all domestic leagues in `master_leagues.json`.

Season format types:
- **`standard`** — UEFA-style split season (e.g. `2025/2026`)
- **`calendar`** — Single calendar year (fetches both `2025` and `2026`)
- **`split`** — Apertura/Clausura (fetches both halves)

Cups and continental competitions are handled in a separate notebook.

In [1]:
import requests
import json
import time
import os

# --- Season format by country code ---
# standard  = 2025/2026 (UEFA and similar)
# calendar  = single year, fetch both 2025 and 2026
# split     = Apertura + Clausura

SEASON_FORMAT = {
    # Standard (2025/2026)
    # "ENG": "standard",
    # "GER": "standard",
    # "FRA": "standard",
    # "ESP": "standard",
    # "ITA": "standard",
    "TUR": "standard",
    "NED": "standard",
    "POR": "standard",
    "BEL": "standard",
    "SCO": "standard",
    "WAL": "standard",
    "IRL": "standard",
    "GRE": "standard",
    "CYP": "standard",
    "CRO": "standard",
    "CZE": "standard",
    "SVK": "standard",
    "SVN": "standard",
    "POL": "standard",
    "HUN": "standard",
    "ROU": "standard",
    "SRB": "standard",
    "BUL": "standard",
    "SUI": "standard",
    "AUT": "standard",
    "RUS": "standard",
    "KAZ": "standard",
    "ARM": "standard",
    "ISR": "standard",
    "KSA": "standard",
    "QAT": "standard",
    "UAE": "standard",
    "IRQ": "standard",
    "IRN": "standard",
    "MAR": "standard",
    "TUN": "standard",
    "ALG": "standard",
    "EGY": "standard",
    "RSA": "standard",
    "GHA": "standard",
    "DEN": "standard",
    "NOR": "standard",
    "SWE": "standard",
    "FIN": "standard",  # may be calendar — check output

    # Calendar (2025 + 2026)
    "BRA": "calendar",
    "ARG": "calendar",
    "COL": "calendar",
    "PAR": "calendar",
    "VEN": "calendar",
    "ECU": "calendar",
    "CHI": "calendar",
    "HON": "calendar",
    "CRC": "calendar",
    "PAN": "calendar",
    "USA": "calendar",  # MLS
    "CAN": "calendar",
    "JPN": "calendar",
    "KOR": "calendar",
    "CHN": "calendar",
    "MAS": "calendar",
    "THA": "calendar",
    "IDN": "calendar",
    "AUS": "calendar",
    "NZL": "calendar",
    "UZB": "calendar",

    # Split (Apertura + Clausura)
    "MEX": "split",
}

# --- Season strings per format ---
SEASONS_FOR_FORMAT = {
    "standard": ["2025/2026"],
    "calendar": ["2025", "2026"],
    "split":    ["2025/2026 - Apertura", "2025/2026 - Clausura"],
}

# --- Output directory ---
OUT_DIR = "fixtures_tiredness"
os.makedirs(OUT_DIR, exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0"}

print("Config loaded.")
print(f"Output directory: {OUT_DIR}/")

Config loaded.
Output directory: fixtures_tiredness/


In [ ]:
# --- Load master leagues (leagues only, no cups, no continentals) ---
with open("TIREDNESS DATASET/   .json") as f:
    all_leagues = json.load(f)

domestic_leagues = [
    l for l in all_leagues
    if l["type"] == "league" and l["country"] != "INT"
]

print(f"Found {len(domestic_leagues)} domestic leagues to scrape\n")

# --- Scrape leagues 61st to 70th (indexes 60 to 69) ---
for league in domestic_leagues[60:70]:
    league_id   = league["id"]
    league_name = league["name"]
    country     = league["country"]
    country_name = league.get("country_name", country)

    fmt = SEASON_FORMAT.get(country, "standard")  # default to standard if unknown
    seasons = SEASONS_FOR_FORMAT[fmt]

    for season in seasons:
        season_slug = season.replace("/", "_").replace(" ", "_").replace("-", "-")
        filename = f"{OUT_DIR}/{country.lower()}_{league_id}_{season_slug}_fixtures.json"

        # Skip if already fetched
        if os.path.exists(filename):
            print(f"  [skip] {country_name} — {league_name} {season} (already exists)")
            continue

        print(f"Fetching {country_name} — {league_name} {season}...", end=" ")

        try:
            r = requests.get(
                f"https://www.fotmob.com/api/data/leagues?id={league_id}&ccode3=USA_NY",
                headers=headers,
                timeout=10
            )

            if r.status_code != 200:
                print(f"✗ HTTP {r.status_code}")
                continue

            data = r.json()

            # Check if the season we want is actually in the response
            selected_season = data.get("details", {}).get("selectedSeason", "")
            all_matches = data.get("fixtures", {}).get("allMatches", [])
            finished = [m for m in all_matches if m["status"].get("finished")]
            upcoming = [m for m in all_matches if not m["status"].get("finished")]

            print(f"✓  season='{selected_season}'  {len(finished)} finished | {len(upcoming)} upcoming")

            # Save with metadata
            output = {
                "meta": {
                    "league_id": league_id,
                    "league_name": league_name,
                    "country": country,
                    "country_name": country_name,
                    "requested_season": season,
                    "selected_season": selected_season,
                    "season_format": fmt,
                    "finished_count": len(finished),
                    "upcoming_count": len(upcoming),
                },
                "data": data
            }

            with open(filename, "w") as f:
                json.dump(output, f)

        except Exception as e:
            print(f"✗ Error: {e}")

        time.sleep(0.3)  # be polite

print("\nDone.")

Found 66 domestic leagues to scrape

Fetching Thailand — Thai League 2025... ✓  season='2025/2026'  240 finished | 0 upcoming
Fetching Thailand — Thai League 2026... ✓  season='2025/2026'  240 finished | 0 upcoming
Fetching Indonesia — Super League 2025... ✓  season='2025/2026'  297 finished | 9 upcoming
Fetching Indonesia — Super League 2026... ✓  season='2025/2026'  297 finished | 9 upcoming
Fetching Finland — Veikkausliiga 2025/2026... ✓  season='2026'  43 finished | 89 upcoming
Fetching Slovakia — 1. liga 2025/2026... ✓  season='2025/2026'  192 finished | 0 upcoming
Fetching Ghana — Premier League 2025/2026... ✓  season='2025/2026'  297 finished | 9 upcoming
Fetching Armenia — Premier League 2025/2026... ✓  season='2025/2026'  126 finished | 9 upcoming

Done.


##

In [12]:
import json
import os

FIXTURES_DIR = "fixtures_tiredness"

results = []

for fname in sorted(os.listdir(FIXTURES_DIR)):
    if not fname.endswith(".json"):
        continue

    with open(os.path.join(FIXTURES_DIR, fname)) as f:
        d = json.load(f)

    meta = d.get("meta", {})
    requested = meta.get("requested_season", "?")
    selected  = meta.get("selected_season", "?")
    mismatch  = "⚠️" if requested != selected else "✓"

    results.append({
        "file":      fname,
        "country":   meta.get("country_name", "?"),
        "league":    meta.get("league_name", "?"),
        "requested": requested,
        "selected":  selected,
        "finished":  meta.get("finished_count", 0),
        "upcoming":  meta.get("upcoming_count", 0),
        "status":    mismatch,
    })

# Print summary
mismatches = [r for r in results if r["status"] == "⚠️"]
print(f"Total files: {len(results)}  |  Mismatches: {len(mismatches)}\n")
print(f"{'Status':<6} {'Country':<20} {'League':<25} {'Requested':<25} {'Selected':<25} {'Fin':>5} {'Up':>5}")
print("-" * 115)
for r in results:
    print(f"{r['status']:<6} {r['country']:<20} {r['league']:<25} {r['requested']:<25} {r['selected']:<25} {r['finished']:>5} {r['upcoming']:>5}")

Total files: 87  |  Mismatches: 33

Status Country              League                    Requested                 Selected                    Fin    Up
-------------------------------------------------------------------------------------------------------------------
✓      Algeria              Ligue 1                   2025/2026                 2025/2026                   218    22
⚠️     Argentina            Liga Profesional          2025                      2026                        254   240
✓      Argentina            Liga Profesional          2026                      2026                        254   240
✓      Armenia              Premier League            2025/2026                 2025/2026                   126     9
⚠️     Australia            A-League                  2025                      2025/2026                   162     1
⚠️     Australia            A-League                  2026                      2025/2026                   162     1
✓      Austria        

In [13]:
import json
import os

FIXTURES_DIR = "fixtures_tiredness"

# Files where 2025 is just a dupe of 2026 (calendar countries that FotMob defaults to current year)
DUPE_2025 = [
    "arg", "bra", "chl", "chn", "ecu", "jpn", "kor", 
    "pan", "pry", "uzb", "ven", "usa"
]

# Countries with both 2025 and 2026 returning same standard season (should only have one file)
DUPE_CALENDAR_AS_STANDARD = [
    "aus", "idn", "mys", "tha", "crc"
]

to_delete = []
to_refetch = []

all_files = sorted(os.listdir(FIXTURES_DIR))

for fname in all_files:
    if not fname.endswith(".json"):
        continue

    country_code = fname.split("_")[0]

    # Dupe 2025 files for pure calendar countries
    if country_code in DUPE_2025 and "_2025_" in fname:
        to_delete.append(fname)

    # Both files are dupes for countries we misclassified as calendar
    if country_code in DUPE_CALENDAR_AS_STANDARD:
        to_delete.append(fname)  # delete both, will re-fetch as standard

# Countries that need a corrected re-fetch
to_refetch = [
    {"country": "Finland",   "code": "fin", "id": None, "format": "calendar"},
    {"country": "Norway",    "code": "nor", "id": None, "format": "calendar"},
    {"country": "Sweden",    "code": "swe", "id": None, "format": "calendar"},
    {"country": "Ireland",   "code": "irl", "id": None, "format": "calendar"},
    {"country": "Kazakhstan","code": "kaz", "id": None, "format": "calendar"},
    {"country": "Honduras",  "code": "hon", "id": None, "format": "split"},
    {"country": "Colombia",  "code": "col", "id": None, "format": "split"},
    {"country": "Australia", "code": "aus", "id": None, "format": "standard"},
    {"country": "Indonesia", "code": "idn", "id": None, "format": "standard"},
    {"country": "Malaysia",  "code": "mys", "id": None, "format": "standard"},
    {"country": "Thailand",  "code": "tha", "id": None, "format": "standard"},
    {"country": "Costa Rica","code": "crc", "id": None, "format": "standard"},
    {"country": "Mexico",    "code": "mex", "id": None, "format": "split",   "note": "Apertura missing"},
    {"country": "New Zealand","code": "nzl", "id": None, "format": "calendar", "note": "2026 not started, 2025 is fine — no action needed"},
]

print("=== FILES TO DELETE ===")
for f in to_delete:
    print(f"  {f}")

print(f"\nTotal to delete: {len(to_delete)}")

print("\n=== LEAGUES TO RE-FETCH ===")
for r in to_refetch:
    note = f"  ← {r['note']}" if r.get('note') else ""
    print(f"  {r['country']:<15} format: {r['format']}{note}")

=== FILES TO DELETE ===
  arg_112_2025_fixtures.json
  aus_113_2025_fixtures.json
  aus_113_2026_fixtures.json
  bra_268_2025_fixtures.json
  chn_120_2025_fixtures.json
  crc_121_2025_fixtures.json
  crc_121_2026_fixtures.json
  ecu_246_2025_fixtures.json
  idn_8983_2025_fixtures.json
  idn_8983_2026_fixtures.json
  jpn_223_2025_fixtures.json
  kor_9080_2025_fixtures.json
  pan_9039_2025_fixtures.json
  tha_8984_2025_fixtures.json
  tha_8984_2026_fixtures.json
  usa_130_2025_fixtures.json
  uzb_540_2025_fixtures.json
  ven_339_2025_fixtures.json

Total to delete: 18

=== LEAGUES TO RE-FETCH ===
  Finland         format: calendar
  Norway          format: calendar
  Sweden          format: calendar
  Ireland         format: calendar
  Kazakhstan      format: calendar
  Honduras        format: split
  Colombia        format: split
  Australia       format: standard
  Indonesia       format: standard
  Malaysia        format: standard
  Thailand        format: standard
  Costa Rica      f